# AI Marketing Budget Decisioning Agents

Multi-agent system: Measurement → Allocation → Explanation → Guardrail

Run all cells to generate a weekly budget decision.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
CHANNELS = ['Meta', 'Google', 'TikTok', 'Programmatic', 'Email', 'Influencer']

def generate_market_state():
    data = []
    for ch in CHANNELS:
        data.append({
            'channel': ch,
            'current_share': np.random.uniform(0.08, 0.35),
            'incremental_roas': np.random.uniform(1.2, 4.5),
            'ltv_index': np.random.uniform(0.7, 1.6),
            'saturation': np.random.uniform(0.3, 0.95),
        })
    df = pd.DataFrame(data)
    df['current_share'] = df['current_share'] / df['current_share'].sum()
    df['expected_value'] = df['incremental_roas'] * df['ltv_index'] * (1 - df['saturation'] * 0.6)
    return df

state = generate_market_state()
state.round(3)

In [ ]:
# Allocation
raw = state['expected_value'].clip(lower=0.1)
raw = raw / raw.sum()
new_share = 0.6 * raw + 0.4 * state['current_share']
new_share = new_share.clip(0.05, 0.40)
new_share = new_share / new_share.sum()
state['proposed_share'] = new_share
state['delta_pp'] = (state['proposed_share'] - state['current_share']) * 100

print(state[['channel', 'current_share', 'proposed_share', 'delta_pp', 'expected_value']].round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(state))
w = 0.35
ax.bar(x - w/2, state['current_share'], w, label='Current', color='#7f8c8d')
ax.bar(x + w/2, state['proposed_share'], w, label='Proposed', color='#2E86AB')
ax.set_xticks(x)
ax.set_xticklabels(state['channel'], rotation=15)
ax.set_ylabel('Budget share')
ax.set_title('Current vs Proposed Budget Allocation')
ax.legend()
plt.tight_layout()
plt.show()

## Explanation & Guardrail
The Explanation Agent turns the numbers into a CMO-ready rationale. The Guardrail Agent flags large swings or constraint violations.